In [ ]:
from collections import Counter
import itertools as it

from ibis import _
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

import src
from src.load import DataLoader

In [ ]:
data = DataLoader()
table = (
    data.comments(filtered=False)
    .join(data.videos(filtered=True), "video_id")
    .join(data.channels(), "channel_id")
    .select(_.video_id, _.comment_author, _.channel)
    .group_by(_.comment_author)
    .agg(channels=_.channel.collect().unique())
)

comments = table.to_pandas()

# Number of distinct commentators by channel

In [ ]:
counts = Counter(
    channel for person_channels in comments.channels.to_list() for channel in person_channels
)

out_df = pd.DataFrame(
    counts.items(),
    columns=["channel", "n_commenters"],
).sort_values("n_commenters", ascending=False)

out_df.to_csv(src.OUT / "tables/n_commenters_per_channel.csv", index=False)

out_df

# Co-Commenting Network

In [ ]:
graph = nx.Graph()
for possible_edges in comments.channels.to_list():
    if len(possible_edges) < 2:
        continue

    for u, v in it.combinations(possible_edges, r=2):
        if not graph.has_edge(u, v):
            graph.add_edge(u, v, weight=1)
        else:
            graph[u][v]["weight"] += 1

In [ ]:
g = graph.copy()
for u, v, data in list(g.edges(data=True)):
    if data["weight"] < 10:
        g.remove_edge(u, v)

pos = nx.spring_layout(g, weight=None, seed=14, iterations=90)
plt.figure(1, figsize=(13, 11))
nx.draw_networkx_nodes(g, pos, node_color="lightblue", node_size=1700)
nx.draw_networkx_edges(g, pos, edge_color="grey", connectionstyle="angle3", arrows=True)
nx.draw_networkx_labels(g, pos, font_size=12, font_family="sans-serif")
nx.draw_networkx_edge_labels(
    g,
    pos,
    edge_labels={(u, v): d["weight"] for u, v, d in g.edges(data=True)},
    connectionstyle="angle3",
)
plt.savefig(src.OUT / "figures/co_commenter_network.svg")
plt.show()